In [31]:
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
from astropy.io.fits import FITS_rec
from tqdm import tqdm

from bloodmoon.mask import CodedMaskCamera, codedmask, count, decode
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.benchmarking import source_catalogue_data
from darksun.data import DataLoader, CatalogueLoader

ds.show.set_figures_darkbkg()

In [32]:
BASE_PATH: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data"
SIM_PATH: str = f"{BASE_PATH}/Simulations"
OUT_PATH: str = f"{BASE_PATH}/Outputs"

# BASE_PATH: str = "/mnt/d/PhD_AASS/Coding/Images_fits"
# SIM_PATH = OUT_PATH = BASE_PATH

In [33]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"
UPS_X: int = 2
UPS_Y: int = 1

wfm: CodedMaskCamera = codedmask(f"{SIM_PATH}/{MASK_FITS}", UPS_X, UPS_Y)

In [34]:
# SKYFIELD: str = "IROSDummy"
# DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"
SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks"

ID_CAMERA_A: str = "cam1a"
DATASET: str = "detected"

filepaths: dict[str, dict[str, Path]] = simulation_files(f"{SIM_PATH}/{SKYFIELD}/{DATA_FITS}")

sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])

In [35]:
type EnergyRange = tuple[float, float]

def get_source_coords(sourceID: str, catalogue: CatalogueLoader) -> CoordEquatorial:
    """Extracts the source RA/Dec coords from catalogue."""
    data = source_catalogue_data(sourceID, catalogue.DLdata)
    return CoordEquatorial(data['RA'], data['DEC'])

def select_target_srcs(
    photons: FITS_rec,
    srcIDs: str | tuple[str, ...],
    catalogue: CatalogueLoader,
) -> FITS_rec:
    """Filters the input photon list for the given sources equatorial coords."""
    srcIDs_ = (srcIDs,) if isinstance(srcIDs, str) else srcIDs
    coords = tuple(get_source_coords(src, catalogue) for src in srcIDs_)
    phs = ds.select_source_photons(coords, photons, False)
    return phs

def select_eband_phs(photons: FITS_rec, eband: EnergyRange) -> FITS_rec:
    """Filters the input photon list in the given energy band."""
    E_min, E_max = eband
    return ds.filter_data(photons, E_min=E_min, E_max=E_max, coords=None)

In [36]:
def analyse_decoding_loss(
    camera: CodedMaskCamera,
    sdl: DataLoader,
    srcID: str,
    ebands: EnergyRange | tuple[EnergyRange, ...],
    catalogue: CatalogueLoader,
) -> tuple[NDArray, NDArray]:
    """Computes the collected photons in given ebands and decoded sky peak for given source."""
    cts: list[float] = []
    peaks: list[float] = []
    ebands_ = (ebands,) if isinstance(ebands[0], (int, float)) else ebands

    photons = select_target_srcs(sdl.DLdata, srcID, catalogue)
    loop = tqdm(ebands_, f'Gathering from {srcID.upper()}')

    for eband in loop:
        phs = select_eband_phs(photons, eband)
        eband_detector, _ = count(camera, phs)
        eband_sky = decode(camera, eband_detector)
        cts.append(eband_detector.sum())
        peaks.append(eband_sky.max())
    
    return map(np.array, (cts, peaks))

In [40]:
scrID: str = 'scox1'
ebands: tuple[EnergyRange] = (
    (2.0, 10.0), (2.0, 3.0), (5.0, 10.0), (10.0, 15.0), (19.0, 20.0), (17.0, 20.0),
)

cts, peaks = analyse_decoding_loss(wfm, sdlA, scrID, ebands, catA)

for eband, c, p in zip(ebands, cts, peaks):
    c, p = map(float, (c, p))
    print(
        f'Eband: {eband} | cts: {c}, peak: {p}, res: {(c - p) * 100 / c:.1f}%, {(c - p) / pow(c, 0.5):.1f}sigma\n'
    )

Gathering from SCOX1: 100%|██████████| 6/6 [00:00<00:00,  7.47it/s]

Eband: (2.0, 10.0) | cts: 1017716.0, peak: 988310.0195895487, res: 2.9%, 29.1sigma

Eband: (2.0, 3.0) | cts: 140475.0, peak: 138645.27906284554, res: 1.3%, 4.9sigma

Eband: (5.0, 10.0) | cts: 392259.0, peak: 373597.3401670377, res: 4.8%, 29.8sigma

Eband: (10.0, 15.0) | cts: 41845.0, peak: 35789.622123533016, res: 14.5%, 29.6sigma

Eband: (19.0, 20.0) | cts: 254.0, peak: 219.14326841682865, res: 13.7%, 2.2sigma

Eband: (17.0, 20.0) | cts: 1212.0, peak: 961.6224737303821, res: 20.7%, 7.2sigma



In [38]:
from bloodmoon.coords import angle2shift 
from bloodmoon.optim import model_sky

PSFY = True if DATASET == 'reconstructed' else False


tx, ty = (12.0, 21.0)  # [deg]
cts = 1e5

sky = model_sky(wfm, angle2shift(wfm, tx), angle2shift(wfm, ty), cts, psfy=PSFY)

cts, sky.max(), (cts - sky.max()) * 100 / cts

(100000.0, np.float64(98454.87690461852), np.float64(1.5451230953814812))